# NSL-KDD — análise exploratória e diagnóstico inicial

Esta é a primeira versão do notebook da etapa 1. O trabalho será ampliado com as decisões discutidas em conjunto.

Segundo os pontos de controle do enunciado:

| Etapa | Foco | Entrega |
| --- | --- | --- |
| 1 | Caracterização, classes, desbalanceamento, categóricos, inconsistências, redundâncias e possíveis vazamentos | 24/10 |
| 2 | Implementação e comparação das técnicas de pré-processamento | 07/11 |
| 3 | Construção e comparação de classificadores | 21/11 |
| 4 | Avaliação final, explicabilidade e apresentação | 28/11 |

Já foi aprovada a comparação entre **manter os registros** e **remover todos os registros dos grupos realmente conflitantes**. Aqui medimos o efeito nos dados. A comparação de desempenho será feita posteriormente.

O alvo oficial ainda não está definido. Os tipos de ataque originais são preservados, e os agrupamentos aparecem somente como cenários de diagnóstico. Este notebook lê apenas `KDDTrain+.txt`.

## 1. Preparar o ambiente e localizar a base

Instale as dependências de `requirements.txt` no ambiente usado pelo notebook. Baixe o ZIP conforme o README e mantenha-o em `data/raw/nslkdd.zip`.

Os resultados com registros da base aparecem apenas ao executar localmente. O notebook é salvo sem saídas, e a base permanece ignorada pelo Git.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "preprocessamento.py").is_file():
    candidato = RAIZ / "novo-projeto"
    if not (candidato / "preprocessamento.py").is_file():
        raise FileNotFoundError("Execute o notebook a partir da pasta do projeto.")
    RAIZ = candidato
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from preprocessamento import (
    ATRIBUTOS, ATRIBUTOS_MODELO, CATEGORICOS, MODOS_ALVO, ESTRATEGIAS,
    carregar_treino, selecionar_entradas, construir_alvo, identificar_conflitos,
    aplicar_estrategia, impacto_por_classe,
)

ARQUIVO_BASE = RAIZ / "data" / "raw" / "nslkdd.zip"
if not ARQUIVO_BASE.is_file():
    raise FileNotFoundError("Baixe a base seguindo o README: data/raw/nslkdd.zip")
print("Versão do pandas:", pd.__version__)

## 2. Ler o treino e identificar o papel das colunas

O TXT não tem cabeçalho. São **41 atributos descritivos**, seguidos do **tipo de ataque** e do **nível de dificuldade**. Portanto, a última coluna não é a classe.

O índice identifica a linha original do TXT, contando a partir de 1. Os nomes dos atributos seguem o esquema da base. A conversão dos campos numéricos serve à leitura: nenhuma normalização ou imputação foi realizada.

In [ ]:
dados = carregar_treino(ARQUIVO_BASE)
X = dados[ATRIBUTOS].copy()
numericos = [coluna for coluna in ATRIBUTOS if coluna not in CATEGORICOS]
print("Registros:", len(dados))
print("Colunas:", dados.shape[1])
print("Atributos descritivos:", len(ATRIBUTOS))
print("Atributos numéricos:", len(numericos))
print("Classe no arquivo:", "attack_type")
print("Metadado separado dos atributos descritivos:", "difficulty")

## 3. Ausências, categorias e atributos constantes

Não confundir zero com ausência: zero pode indicar que determinada ocorrência não aconteceu. A verificação abaixo conta valores ausentes, células vazias e marcadores textuais usuais, sem substituir valores.

As colunas originais permanecem em `X` para o diagnóstico. A decisão aprovada sobre a coluna constante é aplicada separadamente nas entradas dos futuros modelos, na próxima célula.

In [ ]:
marcadores = {"", "?", "na", "nan", "null", "none", "n/a"}
ausencias = dados.isna().sum()
for coluna in CATEGORICOS + ["attack_type"]:
    textos = dados[coluna].astype(str).str.strip().str.lower()
    ausencias.loc[coluna] += textos.isin(marcadores).sum()
display(ausencias.to_frame("ausencias_detectadas"))

categorias = pd.DataFrame({
    "atributo": CATEGORICOS,
    "categorias_no_treino": [dados[c].nunique() for c in CATEGORICOS],
})
display(categorias)

constantes = [c for c in ATRIBUTOS if dados[c].nunique(dropna=False) == 1]
display(pd.DataFrame({"atributo_constante": constantes,
                      "valor": [dados[c].iloc[0] for c in constantes]}))

### Decisão aprovada em 14/10: retirar `num_outbound_cmds` das entradas

O atributo vale **zero nos 125.973 registros de treino**. Sua variância é zero, portanto ele não diferencia nenhuma conexão nesse conjunto.

Decidimos retirar apenas esse atributo das entradas dos futuros modelos. A função `selecionar_entradas` produz uma cópia com **40 atributos**, conservando as linhas, a ordem e os índices. A base original e as tabelas de diagnóstico continuam com os 41 atributos descritivos.

A seleção foi definida com informações do treino e será a mesma em cada conjunto de avaliação. Outras colunas não são removidas automaticamente. Esta é uma simplificação justificada; ela não constitui evidência de melhora de desempenho.

In [ ]:
X_modelo = selecionar_entradas(dados)
print("Atributos no diagnóstico original:", X.shape[1])
print("Atributos nas entradas dos futuros modelos:", X_modelo.shape[1])
print("Registros preservados:", len(X_modelo))
display(pd.DataFrame([{
    "atributo_retirado": "num_outbound_cmds",
    "valor_no_treino": dados["num_outbound_cmds"].iloc[0],
    "valores_distintos_no_treino": dados["num_outbound_cmds"].nunique(),
    "motivo": "Constante em todos os registros de treino",
}]))

## 4. Identificar os rótulos originais e o desbalanceamento

Esta tabela descreve os rótulos encontrados no TXT. Usá-los no diagnóstico não define ainda o alvo oficial da competição. Nenhum balanceamento é aplicado nesta célula.

In [ ]:
frequencias = dados["attack_type"].value_counts().to_frame("registros")
frequencias["percentual"] = (100 * frequencias["registros"] / len(dados)).round(3)
print("Rótulos distintos, incluindo normal:", len(frequencias))
display(frequencias)

## 5. Escalas e valores que precisam ser investigados

Examinar os valores antes de escolher transformação ou remoção. Extremos podem representar eventos reais, inclusive ataques. Neste ponto não removemos outliers nem recodificamos `su_attempted`.

In [ ]:
display(dados[numericos].describe(percentiles=[0.5, 0.95, 0.99]).T)
print("Valores encontrados em su_attempted:")
display(dados["su_attempted"].value_counts().sort_index().to_frame("registros"))

taxas = [c for c in numericos if "rate" in c]
fora_da_faixa = ((dados[taxas] < 0) | (dados[taxas] > 1)).sum()
display(fora_da_faixa.to_frame("taxas_fora_de_0_a_1"))

## 6. Redundância e conflito são verificações diferentes

- Duplicata completa: mesma linha, incluindo classe e dificuldade.
- Repetição com a mesma classe: mesmos 41 atributos e o mesmo rótulo.
- Conflito: mesmos 41 atributos, mas mais de um rótulo.

Dois rótulos distintos para as mesmas entradas representam uma ambiguidade para o classificador. Isso não comprova qual anotação estaria incorreta.

A chave de comparação usa os 41 atributos. A classe e a dificuldade ficam fora dessa chave.

In [ ]:
print("Duplicatas das 43 colunas:", int(dados.duplicated().sum()))
print("Duplicatas dos atributos mais o tipo de ataque:",
      int(dados.duplicated(ATRIBUTOS + ["attack_type"]).sum()))
print("Repetições adicionais somente dos atributos:", int(X.duplicated().sum()))

y_original = construir_alvo(dados, "original")
conflitos_originais = identificar_conflitos(X, y_original)
print("Grupos conflitantes:", len(X.loc[conflitos_originais].drop_duplicates()))
print("Registros envolvidos:", int(conflitos_originais.sum()))

# Exemplos resumidos; o algoritmo comparou todos os 41 atributos.
colunas_exemplo = ["protocol_type", "service", "flag", "src_bytes", "dst_bytes",
                   "attack_type", "difficulty"]
display(dados.loc[conflitos_originais, colunas_exemplo])

## 7. A definição da classe muda a interpretação do conflito

Para entender esse efeito, calculamos três cenários de diagnóstico: tipos originais, famílias e normal/ataque. O [mapeamento original](https://kdd.ics.uci.edu/databases/kddcup99/training_attack_types) coloca `ipsweep`, `nmap` e `portsweep` na família Probe.

Esses cenários não escolhem a configuração oficial. Repetições que ficam com a mesma classe também não são removidas automaticamente.

In [ ]:
cenarios = []
for modo in MODOS_ALVO:
    y_cenario = construir_alvo(dados, modo)
    mascara = identificar_conflitos(X, y_cenario)
    tabela_cenario = X.assign(classe=y_cenario)
    cenarios.append({
        "cenario_diagnostico": modo,
        "classes": int(y_cenario.nunique()),
        "grupos_conflitantes": len(X.loc[mascara].drop_duplicates()),
        "registros_conflitantes": int(mascara.sum()),
        "registros_restantes_apos_exclusao": len(X) - int(mascara.sum()),
        "repeticoes_adicionais_com_mesma_classe": int(tabela_cenario.duplicated().sum()),
    })
resumo_cenarios = pd.DataFrame(cenarios)
display(resumo_cenarios)

## 8. Impacto da remoção por rótulo original

A exclusão pode ser pequena no total e grande em uma classe rara. A tabela mostra quanto cada rótulo original perderia se os grupos conflitantes nesse cenário fossem excluídos.

In [ ]:
impacto_original = impacto_por_classe(y_original, conflitos_originais)
display(impacto_original.loc[impacto_original["removidos"] > 0])

## 9. Preparar as duas versões aprovadas para inspeção

As duas versões abaixo usam os rótulos originais apenas para conferir o comportamento da limpeza. São cópias em memória; o ZIP e o DataFrame original permanecem preservados.

**Estas versões do treino completo não são uma base pré-limpa para validação cruzada.** Ao iniciar os experimentos, a detecção de conflitos precisa ser refeita somente na parte de treino de cada rodada, sem consultar os rótulos da validação.

In [ ]:
versoes_diagnostico = {}
comparacao = []
for estrategia in ESTRATEGIAS:
    X_versao, y_versao, linhas_removidas = aplicar_estrategia(X, y_original, estrategia)
    versoes_diagnostico[estrategia] = (X_versao, y_versao, linhas_removidas)
    comparacao.append({
        "estrategia": estrategia,
        "registros": len(X_versao),
        "registros_removidos": len(linhas_removidas),
        "classes_preservadas": int(y_versao.nunique()),
    })
display(pd.DataFrame(comparacao))

## 10. Separar diagnóstico da futura avaliação

Quando a classe oficial e o esquema de validação forem definidos, o uso em cada rodada será:

```python
X_parte_treino = selecionar_entradas(dados_parte_treino)
X_parte_validacao = selecionar_entradas(dados_parte_validacao)
X_ajustado, y_ajustado, linhas_removidas = aplicar_estrategia(
    X_parte_treino, y_parte_treino, estrategia
)
```

O classificador será ajustado com essa saída. A parte de validação conservará todos os seus registros; seus rótulos serão usados apenas para avaliar as previsões. O teste oficial seguirá reservado à etapa final.

Não foi selecionado um classificador, uma codificação, uma normalização ou um balanceamento neste notebook.

In [ ]:
# Pendente: não define uma classificação oficial durante esta etapa.
ALVO_OFICIAL = None
print("Alvo oficial:", "a definir" if ALVO_OFICIAL is None else ALVO_OFICIAL)
print("Esta versão apresenta diagnóstico e comparação do efeito nos dados.")

## 11. Diagnóstico desta primeira versão e próximos pontos

- Conferir as tabelas de ausências, categorias, constantes, escalas e classes.
- Examinar os grupos conflitantes e a perda relativa nas classes raras.
- A comparação de manter versus excluir foi implementada; ainda não há resultado de desempenho.
- Discutir o papel do nível de dificuldade e a origem de informações que possam causar vazamento.
- Completar a investigação das relações entre atributos e classe antes de treinar modelos, como pede o enunciado. Correlação alta, isoladamente, não demonstra vazamento: é necessário conferir a origem e a disponibilidade da informação.
- A retirada de `num_outbound_cmds` das entradas foi aprovada e implementada; o diagnóstico conserva o atributo original.
- Discutir `su_attempted = 2` antes de decidir qualquer correção.
- Escolher posteriormente as técnicas de codificação, transformação numérica, balanceamento e redução de dimensionalidade ou seleção de instâncias.

O notebook continua em desenvolvimento. Esta primeira versão cobre o diagnóstico inicial e a preparação da comparação dos conflitos.

## Referências

- Enunciado do trabalho: pontos de controle e requisitos de prevenção de vazamento.
- Material da disciplina: *Parte 3 — Processamento — Dados inconsistentes e redundantes* e *Parte 9 — Processamento — Eliminação de outliers*, disponíveis localmente na pasta `matéria`.
- [NSL-KDD no Kaggle](https://www.kaggle.com/datasets/hassan06/nslkdd).
- [Documentação da base na UNB](https://www.unb.ca/cic/datasets/nsl.html).
- [Mapeamento original das famílias](https://kdd.ics.uci.edu/databases/kddcup99/training_attack_types).

O relatório final deverá registrar o uso das ferramentas de apoio conforme o enunciado.